In [5]:
%pip install gradio opencv-python

Note: you may need to restart the kernel to use updated packages.


In [2]:
import torch

device = torch.device("cuda")

# # Check for accelerator availability
if torch.cuda.is_available():
     device = torch.device("cuda")
     print("Using CUDA for training.")
else:
     print("Using CPU for training.")

Using CUDA for training.


In [3]:
import gradio as gr
from torchvision.utils import draw_bounding_boxes, draw_segmentation_masks
from pathlib import Path

from dataset_transforms import get_transforms
from model import get_model_instance_segmentation
from class_dataset import num_classes as class_num_classes

import pandas as pd

# dataset = "instance"
dataset = "class"


if dataset == "instance":
    num_classes = 2
    model_path = "model_instance.pth"
elif dataset == "class":
    num_classes = class_num_classes
    model_path = "model_class.pth"

model = get_model_instance_segmentation(num_classes).to(device)
model.load_state_dict(torch.load(model_path, map_location=device))

eval_transform = get_transforms(is_train=False)


def threshold_predictions(pred, confidence_threshold):
    # Create a mask for scores above the threshold
    score_mask = pred["scores"] > confidence_threshold

    # Filter the labels and boxes using the mask
    filtered_labels = [label for label, keep in zip(pred["labels"], score_mask) if keep]
    filtered_boxes = pred["boxes"][score_mask]

    return {
        "labels": filtered_labels,
        "scores": pred["scores"][score_mask],
        "boxes": filtered_boxes,
        "masks": pred["masks"][score_mask],
    }


def predict(image, confidence_threshold):
    # convert numpy image to tensor
    image = torch.tensor(image).permute(2, 0, 1).float() / 255.0

    model.eval()
    with torch.no_grad():
        x = eval_transform(image)
        # convert RGBA -> RGB and move to device
        x = x[:3, ...].to(device)
        predictions = model([x])
        pred = predictions[0]

    image = (255.0 * (image - image.min()) / (image.max() - image.min())).to(
        torch.uint8
    )
    image = image[:3, ...]

    # Resize image to 2048x2048
    image = torch.nn.functional.interpolate(
        image.unsqueeze(0), size=(2048, 2048), mode="bilinear"
    ).squeeze(0)

    pred = threshold_predictions(pred, confidence_threshold)

    box_labels = [
        f"{label}:{score:.3f}" for label, score in zip(pred["labels"], pred["scores"])
    ]

    boxes = pred["boxes"].long()
    masks = (pred["masks"] > 0.7).squeeze(1)

    output_image = draw_segmentation_masks(image, masks, alpha=0.5, colors="yellow")
    output_image = draw_bounding_boxes(
        output_image,
        boxes,
        box_labels,
        colors="#ee0044",
        width=4,
        font_size=52,
        font=str(Path("../../assets/zed-mono-bold.ttf")),
    )

    np_image = output_image.permute(1, 2, 0).numpy()
    total_colonies = sum([int(label) for label in pred["labels"]])

    count_dict = {}

    label_list = [int(label) for label in pred["labels"]]

    for label in label_list:
        if label in count_dict:
            count_dict[label] += 1
        else:
            count_dict[label] = 1

    # Create a dataframe with x as the label and y as how many times it appears
    df = pd.DataFrame(
        {"Group size": count_dict.keys(), "Occurrences": count_dict.values()}
    )

    return np_image, total_colonies, df


demo = gr.Interface(
    fn=predict,
    inputs=[
        "image",
        gr.Slider(label="Confidence threshold", maximum=1, value=0.9),
    ],
    examples=[
        ["../../data/unlabeled/default_mmrsbpb_default_19012024140158.jpg"],
        ["../../data/unlabeled/default_mmrsbpb_default_19012024141147.jpg"],
        ["../../data/unlabeled/default_mmrsbpb_default_19012024140912.jpg"],
        ["../../data/unlabeled/default_mmrsbpb_default_19012024141651.jpg"],
    ],
    outputs=[
        gr.Image(label="Segmented image"),
        gr.Number(label="Number of colonies detected"),
        gr.BarPlot(label="Group size distribution", x="Group size", y="Occurrences"),
    ],
    title="Colony Detection",
    allow_flagging="never",
)

demo.launch(inline=False, server_name="0.0.0.0")

/home/sf180384/torch/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Running on local URL:  http://0.0.0.0:7861

To create a public link, set `share=True` in `launch()`.
